# LSTM Training + Evaluation Runbook (Colab)
Clone the repo, run training commands, and generate comparison reports from saved metrics/artifacts.

In [ ]:
# Configure your repository URL and branch
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
BRANCH = "main"
PROJECT_DIR = "major-project"

In [ ]:
# Environment setup
!nvidia-smi
!git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}/backend/stockproject
!python -m pip install --upgrade pip
!pip install -r predictor/requirements.txt

## Optional API Keys
Set keys only if you want live sentiment fetching from APIs during runs.

In [ ]:
import os

# Uncomment and set values if needed
# os.environ['NEWS_API_KEY'] = 'YOUR_NEWSAPI_KEY'
# os.environ['GNEWS_API_KEY'] = 'YOUR_GNEWS_KEY'

print('NEWS_API_KEY configured:', bool(os.getenv('NEWS_API_KEY')))
print('GNEWS_API_KEY configured:', bool(os.getenv('GNEWS_API_KEY')))

In [ ]:
# Quick sentiment + preprocessing sanity check
!python - <<'PY'
import sys
sys.path.append('.')
from predictor.asset_aware_trainer import AssetAwareTrainer
from predictor.nse_data_fetcher import NSEDataFetcher

symbol = 'RELIANCE.NS'
trainer = AssetAwareTrainer(seed=42)
fetcher = NSEDataFetcher()
_, asset_type = trainer.get_asset_config(symbol)
df = fetcher.fetch_data(symbol, period='1y')
processed = trainer.preprocess_data_consistent(df, asset_type, symbol=symbol)
non_zero = int((processed['news_sentiment'].abs() > 1e-12).sum())
print('rows:', len(processed))
print('non_zero_sentiment_rows:', non_zero)
print('tail:')
print(processed[['date','news_sentiment']].tail(8).to_string(index=False))
PY

## Training Commands
Run these cells in sequence or selectively as needed.

In [ ]:
# Single-symbol trial run (fast validation)
!python predictor/asset_aware_trainer.py --symbol RELIANCE.NS --model lstm --period 5y --seq-len 120 --seed 42

In [ ]:
# Train LSTM across tracked assets
!python predictor/asset_aware_trainer.py --all --model lstm --period 5y --seq-len 120 --seed 42

In [ ]:
# Train a global LSTM model from pooled data
!python predictor/global_trainer.py --model lstm

In [ ]:
# Warm-start per-symbol LSTM from global pretrained weights
!python predictor/asset_aware_trainer.py --all --model lstm --period 5y --seq-len 120 --seed 42 --init-from-global

## Aggregate Metrics + Leaderboard

In [ ]:
import glob
import json
import os
import pandas as pd

metrics_files = sorted(glob.glob('predictor/models/metrics_*.json'))
rows = []
for path in metrics_files:
    with open(path, 'r', encoding='utf-8') as f:
        m = json.load(f)
    rows.append({
        'symbol': m.get('symbol'),
        'model_type': m.get('model_type'),
        'mae': m.get('metrics', {}).get('mae'),
        'rmse': m.get('metrics', {}).get('rmse'),
        'mape': m.get('metrics', {}).get('mape'),
        'r2': m.get('metrics', {}).get('r2'),
        'directional_accuracy': m.get('metrics', {}).get('directional_accuracy'),
        'naive_mae': m.get('naive_baseline', {}).get('mae'),
        'naive_rmse': m.get('naive_baseline', {}).get('rmse'),
        'naive_mape': m.get('naive_baseline', {}).get('mape'),
        'naive_directional_accuracy': m.get('naive_baseline', {}).get('directional_accuracy')
    })

df = pd.DataFrame(rows)
if df.empty:
    print('No metrics files found. Run training cells first.')
else:
    df['mae_improvement_vs_naive_pct'] = ((df['naive_mae'] - df['mae']) / df['naive_mae']) * 100
    df['rmse_improvement_vs_naive_pct'] = ((df['naive_rmse'] - df['rmse']) / df['naive_rmse']) * 100
    df = df.sort_values(['model_type', 'mae'])
    os.makedirs('predictor/models/reports', exist_ok=True)
    out_csv = 'predictor/models/reports/lstm_leaderboard.csv'
    df.to_csv(out_csv, index=False)
    print(f'Saved: {out_csv}')
    display(df)

In [ ]:
# Quick visual comparison: model MAE vs naive MAE
import matplotlib.pyplot as plt

if 'df' not in globals() or df.empty:
    print('Run the leaderboard cell first.')
else:
    plot_df = df[df['model_type'] == 'lstm'].copy()
    plot_df = plot_df.sort_values('mae')
    plt.figure(figsize=(14, 6))
    plt.plot(plot_df['symbol'], plot_df['mae'], marker='o', label='LSTM MAE')
    plt.plot(plot_df['symbol'], plot_df['naive_mae'], marker='x', label='Naive MAE')
    plt.xticks(rotation=60, ha='right')
    plt.ylabel('MAE')
    plt.title('LSTM vs Naive Baseline (MAE)')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Optional: archive model artifacts for download
!zip -r predictor/models/reports_and_models.zip predictor/models
print('Created predictor/models/reports_and_models.zip')